Subquestion 3: Is the gradual increase of the state pension age (AOW) from 65 to 67 years (2013–2024) associated with an increase in work trips among people aged 65–70, compared with people aged 60–65?

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

path = '../Data_age_research/1_Mobility_trends_by_trip_(84755NED)_complete.csv'

df = pd.read_csv(                      #read the file from line 10 since that is where the data starts
    path, 
    sep=';', 
    skiprows=9, 
    header=None, 
    encoding='cp1252', 
    engine='python', 
    on_bad_lines='skip'
)


headers = pd.read_csv(                 #read the headers of each column from row 5-9
    path, 
    sep=';', 
    skiprows=4, 
    nrows=5, 
    header=None, 
    encoding='cp1252', 
    engine='python', 
    on_bad_lines='skip'
)

#find the columns of the relevant age groups for this research question from the file, and assign the index of relevant columns to a dictionary

relevant_age_groups = ['50 to 60 years', '60 to 65 years', '65 to 70 years', '70 years or older']
column_dictionary = {}               


for column_id in range(1, len(headers.columns)):                      #find the attributes of each column 
    try:
        mode = str(headers.iloc[0, column_id])
        ages = str(headers.iloc[2, column_id])
        motive = str(headers.iloc[3, column_id])
        
        if 'To and from work' in motive and 'Total' in mode:     #assign the column ID to the dictionary if the column 
            for age in relevant_age_groups:                           #contains right age group, commute trips, and all modes      
                if age in ages:
                    column_dictionary[age] = column_id
    except Exception:
        continue

print('Gevonden kolom-indexen:', column_dictionary)


Gevonden kolom-indexen: {'50 to 60 years': 44, '60 to 65 years': 50, '65 to 70 years': 56, '70 years or older': 62}


In [3]:
# create a new dataframe (df_clean) in which we filter the cleaned data
df_clean = pd.DataFrame()
df_clean['Periode'] = df.iloc[:, 0]                                   #make the first column the show the periods

for age, column_id in column_dictionary.items():                      #for all columns with relevant age groups, 
    df_clean[age] = pd.to_numeric(                                    #make all values readable for python
        df.iloc[:, column_id].astype(str).str.replace(',', '.'),      #(all floats with . instead of , for decimals)
        errors='coerce')

df_clean['year_clean'] = pd.to_numeric(                               #make a new column with cleaned up years
    df_clean['Periode'].str.replace('*', '', regex=False), errors='coerce')

df_clean = df_clean.dropna(subset=['year_clean'])                     #delete all rows that didn't contain a valid year
df_clean['year_clean'] = df_clean['year_clean'].astype(int)           #make all years nice integers

df_filtered = df_clean[(df_clean['year_clean'] >= 2005) & (df_clean['year_clean'] <= 2025)]     #take rows from 2005 to 2025


print(df_filtered)

   Periode  50 to 60 years  60 to 65 years  65 to 70 years  70 years or older  \
6    2005*            0.72            0.32            0.14                NaN   
7    2006*            0.75            0.34            0.15               0.07   
8    2007*            0.77            0.36            0.16               0.07   
9    2008*            0.81            0.39            0.17                NaN   
10   2009*            0.81            0.40            0.17               0.07   
11   2010*            0.84            0.44            0.19               0.08   
12   2011*            0.83            0.45            0.18               0.08   
13   2012*            0.82            0.47            0.19               0.08   
14   2013*            0.83            0.50            0.20               0.08   
15   2014*            0.84            0.53            0.21               0.09   
16   2015*            0.83            0.55            0.21               0.09   
17   2016*            0.83  

In [4]:
import plotly.graph_objects as go


fig = go.Figure()                                                             #make new empty figure


styles = {                                                                    #make styles for each group we want to display
    '50 to 60 years': {'dash': 'dash', 'color': 'gray'},
    '60 to 60 years': {'dash': 'solid', 'color': 'blue'},
    '65 to 70 years': {'dash': 'solid', 'color': 'red'},      
    '70 years or older': {'dash': 'dot', 'color': 'green'}
}

# Add a line to the figue for each age group
for age in column_dictionary.keys():
    if age in df_filtered.columns:
        style = styles.get(age, {'dash': 'solid', 'color': 'black'})
        
        fig.add_trace(go.Scatter(
            x=df_filtered['year_clean'],
            y=df_filtered[age],
            mode='lines+markers',
            name=age,
            line=dict(
                color=style['color'], 
                dash=style['dash'], 
                width=3 if '65' in age else 1.5)))

# make a vertical line in 2013 to show the start of the policy change
fig.add_vline(
    x=2013, 
    line_dash="dash", 
    line_color="black", 
    annotation_text="Start policy change (2013)", 
    annotation_position="top left")

# Layout of figure
fig.update_layout(
    title='Mean comute trips per person per day per agegroup(2005–2025)',
    xaxis_title='Year',
    yaxis_title='Trips per person per year',
    template='plotly_white',
    hovermode='x unified', 
    legend=dict(x=0.02, y=0.98),
    width=900,
    height=500)

fig.show()

In [5]:


# Make a copy of the filtered data to use for calculating the indexed figure 
df_indexed = df_filtered.copy()
reference_year = 2013                                            #set 2013, the year introducing the policy, as the reference year

# Calculate index value per age group (value /value in reference year * 100)
for age in column_dictionary.keys():
    if age in df_indexed.columns:
        reference_value = df_indexed.loc[df_indexed['year_clean'] == reference_year, age].values[0]
        
        # Bereken de indexkolom
        df_indexed[age + '_index'] = (df_indexed[age] / reference_value) * 100

# Make a new empty figure to make this index plot
fig_index = go.Figure()

styles = {                                                                      #make styles again
    '50 to 60 years': {'dash': 'dash', 'color': 'gray'},
    '60 to 65 years': {'dash': 'solid', 'color': 'blue'},
    '65 to 70 years': {'dash': 'solid', 'color': 'red'},      
    '70 years or older': {'dash': 'dot', 'color': 'green'}}

for age in column_dictionary.keys():                                            #add agegroups to index lines again      
    if age in df_indexed.columns:
        style = styles.get(age, {'dash': 'solid', 'color': 'black'})
        index_column = age + '_index'
        
        fig_index.add_trace(go.Scatter(
            x=df_indexed['year_clean'],
            y=df_indexed[index_column],
            mode='lines+markers',
            name=f"{age}",
            line=dict(
                color=style['color'], 
                dash=style['dash'], 
                width=3 if '65' in age else 1.5)))

# add horizontal line to help visualize 100
fig_index.add_hline(
    y=100, 
    line_dash="dot", 
    line_color="gray", 
    annotation_text=f"Basisjaar {reference_year} (= 100)", 
    annotation_position="bottom right")

# add vertical line at reference year 2013 to help visualise the start of the policy change
fig_index.add_vline(
    x=2013, 
    line_dash="dash", 
    line_color="black", 
    annotation_text="Start policy change", 
    annotation_position="top left")

# Layout 
fig_index.update_layout(
    title=f'Indexgraph comute trips per person per year (reference year {reference_year} = 100)',
    xaxis_title='year',
    yaxis_title='Index value (2013 = 100)',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(x=0.02, y=0.98),
    width=900,
    height=500)

fig_index.show()

In [10]:
# Split the data into 2 different periods
df_pre = df_filtered[(df_filtered['year_clean'] >= 2005) & (df_filtered['year_clean'] <= 2012)]
df_post = df_filtered[(df_filtered['year_clean'] >= 2013) & (df_filtered['year_clean'] <= 2024)]

# select the relevant age groups
relevant_groups = list(column_dictionary.keys())

#Calculate average comute trips per group
average_before = df_pre[relevant_groups].mean()
average_after = df_post[relevant_groups].mean()

# make a table
analysis_table = pd.DataFrame({
    'Average (2005–2012)': average_before,
    'Average (2013–2024)': average_after
})

# calculate absolute and percentual increase
analysis_table['Absolute increase'] = analysis_table['Average (2013–2024)'] - analysis_table['Average (2005–2012)']
analysis_table['Percentual increase (%)'] = (analysis_table['Absolute increase'] / analysis_table['Average (2005–2012)']) * 100

# Print de table and round to 3 digits
print("AVERAGE COMUNTE TRIPS BEFORE AND AFTER AOW CHANGE")
print(analysis_table.round(3))

AVERAGE COMUNTE TRIPS BEFORE AND AFTER AOW CHANGE
                   Average (2005–2012)  Average (2013–2024)  \
50 to 60 years                   0.794                0.768   
60 to 65 years                   0.396                0.553   
65 to 70 years                   0.169                0.228   
70 years or older                0.075                0.092   

                   Absolute increase  Percentual increase (%)  
50 to 60 years                -0.026                   -3.307  
60 to 65 years                 0.156                   39.432  
65 to 70 years                 0.060                   35.309  
70 years or older              0.017                   23.333  
